# Reverse-QTY series C MD, part 2: the converted dimer in water

Taner Karagol and Alper Karagol.

**Runtime > Change runtime type > GPU, then Run all, then close it.** It stops when
the wall budget is nearly spent and resumes where it left off, so a second Run all
costs about a minute of setup and continues from the last finished chunk.

There are two part 2 notebooks, `1433_rQTY_MD_2_native` and `_2_converted`. They are
independent and can run at the same time in separate sessions. When both have
finished, open `1433_rQTY_MD_3_compare`, which applies the kill criterion.

## The question these two runs answer

The reverse-QTY rule converts every glutamine, threonine and tyrosine inside a helix
and carries no criterion of burial or helical face, so it cannot exclude a dimer
interface. 14-3-3 is the first target of this line whose fold is an obligate dimer.
The interface was measured first, on four structures across three paralogues, and
three of the converted positions lie in it each time, carrying 19 to 25 per cent of
the area buried on dimer formation. Two of the three are the glutamines of the first
helix. The third is the tyrosine whose hydroxyl hydrogen bonds across the interface
to an arginine and an aspartate of the partner subunit, which the tyrosine to
phenylalanine substitution deletes by construction.

The kill criterion for the series, stated before anything was built: the converted
dimer has to hold its interface in water over 100 ns, measured against the native
dimer run in the same conditions, and the series stops if it does not.

The criterion is one-sided below that length, which is what makes a short first look
worth taking. An interface that is coming apart shows it early, so a run of a few
tens of nanoseconds can stop the series. It cannot pass it: not falling apart in 20
ns is not evidence of holding, and the published companion study ran 100 ns. `NS_PER_BUILD`
is set to 20 for this first pass and `TOTAL_NS` to the planned 100: read part 3 when
both builds reach 20, and raise `NS_PER_BUILD` to 100 only if nothing has stopped.
The chunks are cumulative, so continuing costs nothing already spent, and the frame
spacing comes from the planned total, so the early and late parts of a staged run
are sampled alike.

## What it measures

Three observables per frame, all after the burn-in fraction:

- the area buried on dimer formation, from the surface of each subunit alone and of
  the pair, which is the primary quantity the criterion is stated in,
- the number of inter-subunit contacts within 5 angstrom, and the shortest distance
  between the subunits,
- the shortest distance from each converted interface position to the partner
  subunit, which exists in both builds and is therefore the comparable per-position
  measurement,

together with the backbone RMSD, and in the native build the named polar pairs from
the project's own interface analysis. Those pairs have no donor in the converted
build, which is the substitution itself, so the notebook records them as absent
instead of stopping.

Every atom number is resolved by name in the reference frame this build wrote. A
substitution renumbers every atom after it, so a typed serial would measure the
wrong atom in one of the two builds with nothing to warn you.

## What it refuses to do

A folder whose residue at any of the 30 converted positions disagrees with the build
spec, which catches the two CHARMM-GUI uploads going in the wrong order. A topology
carrying a disulfide, since part 1 established this structure has none and a built
bond would hold the fold together by itself. Two builds whose ion counts differ,
which would mean they do not have the same net charge when every substitution here
is neutral to neutral. Two builds whose thermostat or timestep differ. A chunk that
wrote fewer frames than it should, or a concatenation shorter than it reports. A
missing GPU, refused at the start instead of hours later.


In [ ]:
#@title 1. Settings
PROJECT_DIR     = "" #@param {type:"string"}
#@markdown The project folder, used when it is not where DRIVE_ROOT and
#@markdown PROJECT_FOLDER below point. Set it to wherever you put this folder,
#@markdown on Drive or on the Colab disk, and every other path follows from it.
#@markdown Left empty it falls back to those two.
DRIVE_ROOT      = "/content/drive/MyDrive"  #@param {type:"string"}
PROJECT_FOLDER  = "14-3-3-rQTY" #@param {type:"string"}
BUILDS_ROOT     = "/content/drive/MyDrive/MDs/14-3-3-rQTY" #@param {type:"string"}
#@markdown Where the two CHARMM-GUI downloads were unpacked. The folder names in
#@markdown it are not typed here: a build folder is one holding step3_input.pdb,
#@markdown and its build is the word native or converted in its name. The project's
#@markdown own md/ is searched too, and a root that does not exist is
#@markdown skipped, so either location works and this can be left alone.
NS_PER_BUILD    = 20     #@param {type:"integer"}
#@markdown How far to run now. The criterion is stated at 100 ns and is one-sided
#@markdown below it, so a first pass of 20 can stop the series and cannot pass it.
#@markdown Raise this to 100 if part 3 returns CONTINUE; finished chunks are skipped.
TOTAL_NS        = 100    #@param {type:"integer"}
#@markdown The planned total. The trajectory write interval is set from this and not
#@markdown from NS_PER_BUILD, so every chunk is sampled at the same spacing and a
#@markdown staged run does not end up denser at its start than at its end.
NS_PER_CHUNK    = 5      #@param {type:"integer"}
WALL_BUDGET_HOURS = 23.0 #@param {type:"number"}
#@markdown Stops cleanly here; Run all again to continue from the last chunk.
TARGET_FRAMES   = 2000   #@param {type:"integer"}
#@markdown Frames per build. The write interval is recomputed from this,
#@markdown since CHARMM-GUI's default is set for a 1 ns run.
BURN_FRACTION   = 0.2    #@param {type:"number"}
#@markdown Fraction of each trajectory discarded as settling before any number is taken.
CACHE_GROMACS   = True   #@param {type:"boolean"}
#@markdown Keep the compiled CUDA GROMACS on Drive so later sessions skip the build.
GMX_VER         = "2026.3"    #@param {type:"string"}
GMX_SIMD        = "AVX2_256"  #@param ["AVX2_256", "AVX_512", "SSE4.1", "AUTO"]
REF_T_EXPECTED  = 303.15 #@param {type:"number"}
#@markdown The thermostat temperature the builds must carry, used only when part
#@markdown 1's build spec predates its ref_t_K key; the spec's value wins when it
#@markdown is there. 303.15 K is the value the APOE reverse-QTY paper ran at.
MAXWARN         = 10     #@param {type:"integer"}

import os, time
T_START = time.time()
PROJECT = PROJECT_DIR.strip() or os.path.join(DRIVE_ROOT, PROJECT_FOLDER)
MD = os.path.join(PROJECT, "md")
BUILD = "converted"
BUILD_ORDER = ["native", "converted"]
assert BUILD in BUILD_ORDER
print("this notebook runs build:", BUILD.upper())
print("project      ", PROJECT)
if NS_PER_BUILD > TOTAL_NS:
    raise SystemExit("STOP: NS_PER_BUILD is %g and TOTAL_NS is %g. The write interval "
                     "comes from the planned total, so the total cannot be the "
                     "smaller of the two." % (NS_PER_BUILD, TOTAL_NS))
print("sampling     ", NS_PER_BUILD, "ns now, of a planned", TOTAL_NS, "ns")
print("wall budget  ", WALL_BUDGET_HOURS, "hours")


In [ ]:
#@title 2. Drive, GPU and GROMACS
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, re, subprocess, shutil, datetime, math, collections
import numpy as np

PREFIX = "/usr/local/gromacs"
os.environ["PATH"] = PREFIX + "/bin:" + os.environ["PATH"]
RESULTS = os.path.join(MD, "results")
INPUTS = os.path.join(MD, "inputs")
DATA = os.path.join(PROJECT, "data")
os.makedirs(RESULTS, exist_ok=True)

SPEC_PATH = os.path.join(INPUTS, "build_spec.json")
if not os.path.isfile(SPEC_PATH):
    raise SystemExit("STOP: no %s. Run part 1 first; this notebook reads the build "
                     "spec it writes, so the two cannot disagree." % SPEC_PATH)
SPEC = json.load(open(SPEC_PATH))
POSITIONS = SPEC["positions"]
CONV_IFACE = SPEC["converted_in_interface"]
if sorted(set(BUILD_ORDER)) != sorted(set(SPEC["builds"])):
    raise SystemExit("STOP: part 1 wrote the builds %s and this notebook expects %s"
                     % (SPEC["builds"], BUILD_ORDER))
print("build spec: %s %s on %s, %d converted positions, %d of them in the interface %s"
      % (SPEC["paralogue"], SPEC["accession"], SPEC["pdb_id"], len(POSITIONS),
         len(CONV_IFACE), CONV_IFACE))

REF_T_WANT = SPEC.get("ref_t_K")
REF_T_SOURCE = "part 1 build spec"
if REF_T_WANT is None:
    REF_T_WANT = REF_T_EXPECTED
    REF_T_SOURCE = "this notebook; the spec predates its ref_t_K key"
REF_T_WANT = float(REF_T_WANT)
print("thermostat: %g K expected, from %s" % (REF_T_WANT, REF_T_SOURCE))

IFACE_PATH = os.path.join(DATA, "iface_1433.json")
if not os.path.isfile(IFACE_PATH):
    raise SystemExit("STOP: no %s. It is written by code/iface_1433.py in this "
                     "project, so rerun that and upload the project folder again."
                     % IFACE_PATH)
_ifj = json.load(open(IFACE_PATH))
_hit = [r for r in _ifj["structures"] if r["pdb"] == SPEC["pdb_id"]]
if not _hit:
    raise SystemExit("STOP: %s carries no entry for %s, so its interface contacts "
                     "are unknown" % (IFACE_PATH, SPEC["pdb_id"]))
POLAR = {k: v for k, v in _hit[0]["polar_contacts"].items() if v}
if sorted(int(k) for k in _hit[0]["polar_contacts"]) != sorted(CONV_IFACE):
    raise SystemExit("STOP: the interface analysis lists contacts for %s and the "
                     "build spec says the converted interface positions are %s"
                     % (sorted(_hit[0]["polar_contacts"]), CONV_IFACE))
print("polar contacts across the interface, from the project's own analysis:")
for k in sorted(POLAR, key=int):
    for c in POLAR[k]:
        print("   position %s %s to %s %s at %.2f A in the crystal"
              % (k, c["atom"], c["partner"], c["partner_atom"], c["distance"]))

CG_MARKER = "step3_input.pdb"
CG_REQUIRED = ["step3_input.gro", "step3_input.pdb", "step3_input.psf", "topol.top",
               "index.ndx", "step4.0_minimization.mdp", "step5_production.mdp"]


def find_run_dirs(roots, builds):
    found = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            if CG_MARKER in filenames:
                if dirpath not in [d for _, d, _ in found]:
                    found.append((root, dirpath,
                                  os.path.relpath(dirpath, root).lower()))
                dirnames[:] = []
    out = {}
    for b in builds:
        per_root = [(r, [d for rr, d, rel in found if rr == r and b in rel])
                    for r in roots]
        per_root = [(r, h) for r, h in per_root if h]
        if not per_root:
            raise SystemExit(
                "STOP: no folder holding a %s carries the word %s below any root "
                "searched, %s. Every folder holding a %s: %s."
                % (CG_MARKER, b, roots, CG_MARKER,
                   [d for _, d, _ in found] or "none"))
        root, hits = per_root[0]
        if len(hits) != 1:
            raise SystemExit(
                "STOP: %d folders under %s carry the word %s, %s. One per build is "
                "needed and this is not guessed." % (len(hits), root, b, sorted(hits)))
        out[b] = hits[0]
        for _, other in per_root[1:]:
            for o in other:
                for f in ("step3_input.pdb", "step3_input.gro"):
                    pa, po = os.path.join(out[b], f), os.path.join(o, f)
                    a = os.path.getsize(pa) if os.path.isfile(pa) else None
                    c = os.path.getsize(po) if os.path.isfile(po) else None
                    if a != c:
                        raise SystemExit(
                            "STOP: the %s build is in two places and the copies are "
                            "not the same system. %s has a %s of %s bytes and %s has "
                            "%s. Delete whichever is stale; which one runs is not "
                            "guessed." % (b, out[b], f, a, o, c))
                print("  note: %s is also at %s, same size, and that copy is not used"
                      % (b, o))
    return out


RUN_DIR = find_run_dirs([BUILDS_ROOT, MD], BUILD_ORDER)
print("\nCHARMM-GUI folders:")
for b in BUILD_ORDER:
    miss = [f for f in CG_REQUIRED if not os.path.isfile(os.path.join(RUN_DIR[b], f))]
    extra = "" if not miss else "   MISSING %s" % miss
    print("  %-10s %s%s" % (b, RUN_DIR[b], extra))
    if miss:
        raise SystemExit("STOP: %s is missing %s, so it is not a finished CHARMM-GUI "
                         "GROMACS folder. Unpack the download again and upload the "
                         "whole gromacs/ folder, toppar included." % (RUN_DIR[b], miss))
if not os.path.isdir(os.path.join(RUN_DIR[BUILD_ORDER[0]], "toppar")):
    raise SystemExit("STOP: no toppar/ beside %s. topol.top includes its .itp files by "
                     "relative path, so grompp fails without it."
                     % os.path.basename(RUN_DIR[BUILD_ORDER[0]]))

print("\n--- GPU ---")
HAVE_GPU = False
GPU_NAME = None
try:
    _g = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                         "--format=csv,noheader"], check=False, text=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if _g.returncode == 0 and (_g.stdout or "").strip():
        HAVE_GPU = True
        GPU_NAME = _g.stdout.strip().split("\n")[0]
        print(GPU_NAME)
        print("Chunk times scale with this card, so record it with the run: the same "
              "system takes several times longer on a T4 than on an A100.")
    else:
        print("nvidia-smi exited %d, so the runtime claims a GPU it cannot use."
              % _g.returncode)
        print((_g.stdout or "").strip()[:400])
except FileNotFoundError:
    print("no nvidia-smi, so this runtime has no GPU.")
    print("That is fine for the salvage and compare notebooks: they only call "
          "trjconv, trjcat, distance and rms, all of which run on the CPU.")
    print("A production run needs one. Runtime > Change runtime type > T4 GPU.")


def sh(cmd, cwd=None, label=""):
    print("\n$ " + cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = (r.stdout or "").split("\n")
    if len(lines) > 60:
        print("\n".join(lines[:25]))
        print("   ... %d lines omitted ..." % (len(lines) - 45))
        print("\n".join(lines[-20:]))
    else:
        print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError("%s FAILED (exit %d). The output above is the reason."
                           % (label or cmd, r.returncode))


def gmx_version():
    try:
        return subprocess.run(["gmx", "--version"], capture_output=True, text=True,
                              timeout=90).stdout
    except Exception:
        return ""


CACHE = os.path.join(PROJECT, "gromacs_cuda_%s.tar.gz" % GMX_VER)
CACHE_STALE = False

if "CUDA" in gmx_version():
    print("\nCUDA gmx already present in this session")
elif CACHE_GROMACS and os.path.isfile(CACHE):
    print("\nrestoring the cached GROMACS from", CACHE)
    try:
        sh("tar xzf '%s' -C /" % CACHE, label="restore cache")
    except Exception as e:
        print("the cache could not be read (%s); another session may be writing it. "
              "Building from source instead." % e)
        shutil.rmtree(PREFIX, ignore_errors=True)
    if "CUDA" not in gmx_version():
        print("the cached build does not run here, rebuilding from source")
        shutil.rmtree(PREFIX, ignore_errors=True)
        CACHE_STALE = True

if "CUDA" not in gmx_version():
    nvcc = subprocess.run("which nvcc", shell=True, capture_output=True,
                          text=True).stdout.strip()
    if not nvcc:
        raise SystemExit("STOP: nvcc not found, so there is no CUDA toolkit to build "
                         "against. Set the runtime to GPU and press Run all again.")
    print("\nbuilding GROMACS %s. 20 to 40 minutes, once per Drive folder." % GMX_VER)
    sh("apt-get -qq update && apt-get -qq install -y cmake build-essential "
       "libfftw3-dev", label="apt")
    if not os.path.isdir("/content/gromacs-%s" % GMX_VER):
        sh("wget -q https://ftp.gromacs.org/gromacs/gromacs-%s.tar.gz && "
           "tar xzf gromacs-%s.tar.gz" % (GMX_VER, GMX_VER), cwd="/content",
           label="download")
    os.makedirs("/content/gromacs-%s/build" % GMX_VER, exist_ok=True)
    sh("cmake .. -DGMX_BUILD_OWN_FFTW=OFF -DGMX_GPU=CUDA -DGMX_SIMD=%s "
       "-DCMAKE_INSTALL_PREFIX=%s -DGMX_DOUBLE=OFF -DBUILD_TESTING=OFF"
       % (GMX_SIMD, PREFIX), cwd="/content/gromacs-%s/build" % GMX_VER, label="cmake")
    sh("make -j$(nproc) && make install", cwd="/content/gromacs-%s/build" % GMX_VER,
       label="make")
    if "CUDA" not in gmx_version():
        raise SystemExit("STOP: the build finished but gmx reports no CUDA support.")
    if CACHE_GROMACS and (CACHE_STALE or not os.path.isfile(CACHE)):
        print("\ncaching the build to Drive so later sessions skip it")
        tmp = "%s.%d.tmp" % (CACHE, os.getpid())
        try:
            sh("tar czf '%s' %s" % (tmp, PREFIX.lstrip("/")), cwd="/", label="cache")
            os.replace(tmp, CACHE)
        except Exception as e:
            print("could not cache the build (%s); this costs the next session a "
                  "rebuild and nothing else." % e)
            if os.path.isfile(tmp):
                os.remove(tmp)
    elif CACHE_GROMACS:
        print("\nanother session already cached a build, leaving it alone")

print("\n" + gmx_version().split("\n")[0])

In [ ]:
#@title 3. Helpers
AA = set("ALA ARG ASN ASP CYS GLN GLU GLY HIS HSD HSE HSP ILE LEU LYS MET PHE PRO "
         "SER THR TRP TYR VAL".split())


def gmx(*args, label="", stdin_text=None, quiet=False, check=True):
    cmd = ["gmx"] + [str(a) for a in args]
    if not quiet:
        print("$ " + " ".join(cmd), flush=True)
    r = subprocess.run(cmd, text=True, input=stdin_text,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = r.stdout or ""
    lines = out.split("\n")
    if not quiet:
        if len(lines) > 80:
            print("\n".join(lines[:30]))
            print("   ... %d lines omitted ..." % (len(lines) - 60))
            print("\n".join(lines[-30:]))
        else:
            print(out)
    if r.returncode != 0 and check:
        if quiet:
            print(out[-3000:])
        raise RuntimeError("%s failed (exit %d). The output above is the reason."
                           % (label or cmd[1], r.returncode))
    return r.returncode, out


def must_exist(path, what):
    if not os.path.isfile(path):
        raise RuntimeError("%s reported success but %s was not written." % (what, path))


def read_pdb_atoms(path):
    """-> list of (serial, segid, resid, resname, atomname)."""
    out = []
    for L in open(path, errors="replace"):
        if not L.startswith(("ATOM", "HETATM")):
            continue
        seg = L[72:76].strip() or L[21].strip() or "?"
        out.append((int(L[6:11]), seg, int(L[22:26]), L[17:20].strip(),
                    L[12:16].strip()))
    return out


def protein_segments(atoms):
    """Segments holding amino acids.

    gmx does not always carry the CHARMM-GUI segid through to its output PDB, in which
    case read_pdb_atoms falls back to the chain letter and a 'PROA' prefix test
    silently finds nothing. A residue-name test is true of either spelling.
    """
    rn = {}
    for ser, seg, ri, name, an in atoms:
        rn.setdefault((seg, ri), name)
    return sorted({seg for (seg, ri), name in rn.items() if name in AA})


def progress(**kw):
    """Rewritten after every stage, so a dead session still says where it got to."""
    p = os.path.join(RESULTS, "progress_%s.json" % BUILD)
    cur = json.load(open(p)) if os.path.isfile(p) else {}
    cur.update(kw)
    cur["updated"] = datetime.datetime.now().isoformat(timespec="seconds")
    cur["elapsed_hours"] = round((time.time() - T_START) / 3600.0, 2)
    json.dump(cur, open(p, "w"), indent=2)
    return cur


def budget_left():
    return WALL_BUDGET_HOURS - (time.time() - T_START) / 3600.0


def xtc_frames(path):
    """Frame count straight from gmx check.

    An absent or unreadable file returns None, which callers treat as "re-run
    this chunk and everything after it". Output that gmx produced but this
    function cannot parse stops instead, rather than silently re-running every
    chunk of every build on a paid GPU.

    The count is read from the Coords row of gmx check's summary table, with the
    Last frame line as a second chance if that table is ever renamed.
    """
    if not os.path.isfile(path):
        return None
    r = subprocess.run([os.path.join(PREFIX, "bin", "gmx"), "check", "-f", path],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = r.stdout or ""
    m = re.search(r"^\s*Coords\s+(\d+)", out, re.M)
    if m:
        return int(m.group(1))
    m = re.search(r"Last frame\s+(\d+)", out)
    if m:
        return int(m.group(1)) + 1
    if r.returncode != 0:
        return None
    raise SystemExit(
        "STOP: gmx check read %s and exited 0, but its output carried neither a "
        "Coords row nor a Last frame line, so the frame count is unknown. Fix "
        "this parse rather than letting every chunk be re-run as unsound. The "
        "first lines of the output were:\n%s"
        % (os.path.basename(path), "\n".join(out.split("\n")[:12])))


def xtc_last_time(path):
    """Time stamp of the last frame, in ps. None when unreadable."""
    if not os.path.isfile(path):
        return None
    r = subprocess.run([os.path.join(PREFIX, "bin", "gmx"), "check", "-f", path],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    t = None
    for L in (r.stdout or "").split("\n"):
        m = re.search(r"frame\s+\d+\s+time\s+([-\d.eE+]+)", L)
        if m:
            t = float(m.group(1))
    return t


def want_frames(mdp, ns):
    """Frames a chunk of ns nanoseconds should hold, read from the mdp."""
    dt = float(mdp_get(mdp, "dt"))
    every = int(mdp_get(mdp, "nstxout-compressed"))
    return int(round(ns * 1000.0 / dt / every))


def ndx_args(group="SOLU"):
    """-n index.ndx, with a clear stop when it is not there.

    SOLU and SOLV are CHARMM-GUI's own groups and exist only in index.ndx. Given
    -s and no -n, trjconv and rms build the default groups from the topology,
    which hold System, Protein, the side chain sets and the residue names, and no
    SOLU. The selection then fails as "Cannot read from input" after the group
    list has scrolled past, so the failure is easy to misread. grompp is not
    affected because every grompp call here passes -n.
    """
    if not os.path.isfile("index.ndx"):
        raise SystemExit(
            "STOP: no index.ndx in %s, so the group %s cannot be resolved. It comes "
            "from the CHARMM-GUI download and is staged in with the other inputs, so "
            "if it is missing here the staging is wrong."
            % (os.getcwd(), group))
    return ("-n", "index.ndx")


def stage_in(run, work, n_chunks, per_chunk):
    """Copy inputs and any sound restart files from Drive onto local VM disk.

    Everything heavy then runs on local disk. A concatenated trajectory written
    straight into the Drive mount leaves gigabytes of upload in flight, and any
    part still unwritten when the runtime ends comes back truncated.

    A chunk is reusable only if every chunk before it is. Re-running chunk k
    produces a different trajectory from k onward, so a later chunk that came off
    the old branch cannot be stitched onto the new one.
    """
    os.makedirs(work, exist_ok=True)
    heavy = ("prod_all.xtc", "prod_nojump.xtc", "prod_fit.xtc")
    for name in sorted(os.listdir(run)):
        src = os.path.join(run, name)
        if os.path.isdir(src):
            if name == "toppar":
                shutil.copytree(src, os.path.join(work, name), dirs_exist_ok=True)
            continue
        if re.match(r"prod_\d\d\.xtc$", name) or name in heavy:
            continue
        shutil.copy2(src, os.path.join(work, name))
    kept = 0
    for i in range(1, n_chunks + 1):
        nm = "prod_%02d" % i
        src = os.path.join(run, nm + ".xtc")
        if not (os.path.isfile(src) and os.path.isfile(os.path.join(work, nm + ".gro"))):
            print("  %s absent, so it and every chunk after it is re-run" % nm)
            break
        shutil.copy2(src, os.path.join(work, nm + ".xtc"))
        got = xtc_frames(os.path.join(work, nm + ".xtc"))
        if got is None or got < per_chunk - 1:
            os.remove(os.path.join(work, nm + ".xtc"))
            print("  %s holds %s of %d frames, so it and every chunk after it is "
                  "re-run" % (nm, got, per_chunk))
            break
        kept += 1
    for i in range(kept + 1, n_chunks + 1):
        for ext in (".xtc", ".gro", ".cpt", "_prev.cpt", ".tpr", ".log", ".edr"):
            f = os.path.join(work, "prod_%02d%s" % (i, ext))
            if os.path.isfile(f):
                os.remove(f)
    print("  staged into %s, %d of %d chunks reusable" % (work, kept, n_chunks))
    return kept


DRIVE_PENDING = [0]       # bytes handed to the Drive mount and not yet waited on
DRIVE_SENT = {}           # drive path -> bytes written, for the check at the end


def stage_out(work, run, names):
    """Copy products into the Drive mount and count the bytes now in flight.

    The size check below compares the destination with the source, and on a FUSE
    mount that compares the local cache against the local file. It catches a short
    write and says nothing at all about whether Google has the bytes, so a
    truncated upload passes it. What
    gives the upload time is drive_wait, and the only real proof is the next
    session's stage_in, which counts frames in the copy it pulls back down.
    """
    for nm in names:
        src = os.path.join(work, nm)
        if not os.path.isfile(src):
            continue
        dst = os.path.join(run, nm)
        shutil.copy2(src, dst)
        got, want = os.path.getsize(dst), os.path.getsize(src)
        if got != want:
            raise SystemExit("STOP: %s is %d bytes in the Drive mount against %d on "
                             "disk, so even the cached copy is short."
                             % (nm, got, want))
        DRIVE_PENDING[0] += want
        DRIVE_SENT[dst] = want


def drive_wait(why="", floor=20.0, mbps=12.0):
    """Give the Drive daemon time to finish the uploads just handed to it.

    Colab's mount writes to a local cache and uploads in the background, so a
    runtime that ends with uploads in flight leaves truncated files behind. There
    is no way to ask Google what it holds without unmounting, so this waits on a
    throughput estimate with a floor. It is a margin, not a proof. Twenty seconds
    against an eighty minute chunk is not worth economising on.
    """
    mb = DRIVE_PENDING[0] / 1e6
    DRIVE_PENDING[0] = 0
    if mb <= 0:
        return
    secs = max(floor, mb / mbps)
    print("  waiting %.0f s for Drive to take %.0f MB%s"
          % (secs, mb, " after " + why if why else ""), flush=True)
    os.sync()
    time.sleep(secs)


def drive_verify():
    """Flush the mount, bring it back, and compare every file this run uploaded.

    This is the only check in the session that asks Google rather than the local
    cache. flush_and_unmount makes Colab finish its pending writes, and the sizes
    read after the remount come from Drive's own metadata instead of from a page
    that may never have been sent. It runs last, with every file copied and every
    report written, so if the remount ever stalls it costs an open tab and nothing
    else.
    """
    try:
        from google.colab import drive as _drive
        os.sync()
        print("\nflushing the Drive mount, then checking what Drive actually holds")
        _drive.flush_and_unmount()
        _drive.mount("/content/drive")
    except Exception as _e:
        print("\ncould not flush and remount Drive (%s), so the uploads are "
              "unverified. Leave this tab open for a couple of minutes before "
              "closing it." % _e)
        return
    bad, n = [], 0
    for path, want in sorted(DRIVE_SENT.items()):
        n += 1
        got = os.path.getsize(path) if os.path.isfile(path) else None
        if got != want:
            bad.append((os.path.basename(path), got, want))
    if bad:
        for nm, got, want in bad:
            print("   %-22s %s bytes on Drive against %d written" % (nm, got, want))
        raise SystemExit(
            "STOP: %d of %d files are short or missing on Drive after the flush, so "
            "the upload did not finish. The frames are not lost: open this notebook "
            "again and press Run all, which re-sends what is short and skips every "
            "chunk that is whole." % (len(bad), n))
    print("Drive holds all %d uploaded files at full size. Safe to close the runtime."
          % n)


print("helpers ready, MAXWARN =", MAXWARN)

In [ ]:
#@title 4. Run the build. This is the long cell.
def check_identity(run, build):
    """The folder has to be the build it claims to be, before anything runs."""
    PDB = read_pdb_atoms(os.path.join(run, "step3_input.pdb"))
    with open(os.path.join(run, "step3_input.gro"), errors="replace") as fh:
        fh.readline()
        n_gro = int(fh.readline())
    if len(PDB) != n_gro:
        raise SystemExit("STOP: %s step3_input.pdb has %d atoms and the .gro has %d. "
                         "They must be the same build." % (build, len(PDB), n_gro))
    RES = {}
    for ser, seg, ri, name, an in PDB:
        RES.setdefault((seg, ri), name)
    prot = protein_segments(PDB)
    if len(prot) != len(SPEC["chains"]):
        raise SystemExit("STOP: %s has %d protein segments (%s) and the build spec "
                         "has %d chains. The interface this measures is between two "
                         "subunits, so a monomer or a trimer answers a different "
                         "question." % (build, len(prot), ",".join(prot),
                                        len(SPEC["chains"])))
    want = SPEC["expect"] if build == "native" else SPEC["converted_to"]
    bad = []
    for seg in prot:
        for p in POSITIONS:
            got = RES.get((seg, p))
            if got != want[str(p)]:
                bad.append((seg, p, got, want[str(p)]))
    if bad:
        raise SystemExit("STOP: %s carries the wrong residue at %d of the %d converted "
                         "positions, first %s. This is not the build you think it is, "
                         "or the two CHARMM-GUI uploads went in the wrong order."
                         % (build, len(bad), len(POSITIONS) * len(prot), bad[:4]))
    print("  identity ok: %s at all %d converted positions on %s, %d atoms"
          % ("native residues" if build == "native" else "converted residues",
             len(POSITIONS), ",".join(prot), len(PDB)))
    return len(PDB), prot


def psf_disulfides(run):
    """Every SG to SG bond in the PSF, read back instead of assumed.

    This family has free cysteines and no disulfide, and part 1 refuses a structure
    where any SG pair is within 2.5 angstrom. A bond here would mean CHARMM-GUI
    built one anyway, which would hold the fold together by something the real
    protein does not have.
    """
    path = os.path.join(run, "step3_input.psf")
    if not os.path.isfile(path):
        raise SystemExit("STOP: no %s, so the disulfides cannot be verified" % path)
    lines = open(path, errors="replace").read().splitlines()
    i = next(k for k, l in enumerate(lines) if "!NATOM" in l)
    nat = int(lines[i].split()[0])
    sg = {}
    for l in lines[i + 1:i + 1 + nat]:
        f = l.split()
        if len(f) >= 5 and f[4] == "SG" and f[3] == "CYS":
            sg[int(f[0])] = (f[1], int(f[2]))
    j = next(k for k, l in enumerate(lines) if "!NBOND" in l)
    nb = int(lines[j].split()[0])
    tok = []
    for l in lines[j + 1:]:
        tok += [int(x) for x in l.split()]
        if len(tok) >= 2 * nb:
            break
    return sorted({tuple(sorted((sg[a], sg[b])))
                   for a, b in zip(tok[0::2], tok[1::2]) if a in sg and b in sg})


def interface_groups(build):
    """Index groups for the interface measurements, resolved by name.

    A substitution renumbers every atom after it, so the same measurement is a
    different serial in each build. Nothing here is a typed atom number: the two
    subunits, the three converted interface positions and the polar partners of
    the reference structure are all looked up in the reference frame this build
    actually wrote.
    """
    REF = read_pdb_atoms("prod_ref.pdb")
    prot = protein_segments(REF)
    if len(prot) != 2:
        raise SystemExit("STOP: prod_ref.pdb has %d protein segments, expected 2"
                         % len(prot))
    BY, RN, SEG = {}, {}, collections.defaultdict(list)
    drift = 0
    for i, (ser, seg, ri, name, an) in enumerate(REF, 1):
        if ser != i:
            drift += 1
        BY[(seg, ri, an)] = i
        RN.setdefault((seg, ri), name)
        if name in AA:
            SEG[seg].append(i)
    if drift:
        print("  note: %d of %d atoms carry a serial that differs from their position "
              "in the file; the index groups use the position, which is what gmx reads"
              % (drift, len(REF)))
    groups = collections.OrderedDict()
    groups["sub1"] = SEG[prot[0]]
    groups["sub2"] = SEG[prot[1]]
    groups["dimer"] = sorted(SEG[prot[0]] + SEG[prot[1]])
    for p in CONV_IFACE:
        for k, seg in enumerate(prot, 1):
            ser = [s for (g, ri, an), s in BY.items() if g == seg and ri == p]
            if not ser:
                raise SystemExit("STOP: no residue %d in segment %s of prod_ref.pdb"
                                 % (p, seg))
            groups["pos%d_sub%d" % (p, k)] = sorted(ser)
    pairs = []
    for p, contacts in POLAR.items():
        for c in contacts:
            for k, seg in enumerate(prot, 1):
                other = prot[1] if seg == prot[0] else prot[0]
                a = BY.get((seg, int(p), c["atom"]))
                pr = int(re.sub(r"\D", "", c["partner"]))
                b = BY.get((other, pr, c["partner_atom"]))
                if a is None or b is None:
                    print("     %s %s%s to %s %s not present in this build"
                          % (seg, p, c["atom"], c["partner"], c["partner_atom"]))
                    continue
                nm = "hb_%s_%s_sub%d" % (p, c["partner_atom"], k)
                pairs.append((nm, "%s%s%s to %s %s" % (seg, p, c["atom"],
                                                       c["partner"], c["partner_atom"]),
                              a, b))
    with open("iface.ndx", "w") as fh:
        for nm, ser in groups.items():
            fh.write("[ %s ]\n" % nm)
            for i in range(0, len(ser), 15):
                fh.write(" ".join(str(x) for x in ser[i:i + 15]) + "\n")
        for nm, pretty, a, b in pairs:
            fh.write("[ %s ]\n%d %d\n" % (nm, a, b))
    print("  index groups: 2 subunits of %d and %d atoms, their union at %d, %d "
          "converted interface positions over both subunits, %d polar pairs present"
          % (len(groups["sub1"]), len(groups["sub2"]), len(groups["dimer"]),
             len(CONV_IFACE) * 2, len(pairs)))
    return groups, pairs, prot


def xvg(path, what):
    must_exist(path, what)
    D = np.loadtxt(path, comments=["#", "@"])
    if D.ndim == 1:
        D = D.reshape(-1, len(D)) if D.size else D.reshape(0, 0)
    if not D.size:
        raise SystemExit("STOP: %s is empty, so %s produced no data" % (path, what))
    return D


def sasa(selection, out, label):
    gmx("sasa", "-s", "prod_01.tpr", "-f", "prod_fit.xtc", "-n", "iface.ndx",
        "-surface", selection, "-o", out, "-xvg", "none", label=label, quiet=True)
    return xvg(out, label)[:, 1]


def measure(run, build, tpr, groups, pairs, prot, natoms):
    """The three observables the kill criterion is stated in, per frame."""
    burn = None
    ab = sasa('group "dimer"', "sasa_dimer.xvg", "sasa dimer")
    a = sasa('group "sub1"', "sasa_sub1.xvg", "sasa subunit 1")
    b = sasa('group "sub2"', "sasa_sub2.xvg", "sasa subunit 2")
    n = min(len(ab), len(a), len(b))
    if n < 10:
        raise SystemExit("STOP: the surface series hold %d frames between them" % n)
    buried = (a[:n] + b[:n] - ab[:n]) * 100.0 / 2.0
    burn = int(n * BURN_FRACTION)

    gmx("mindist", "-s", tpr, "-f", "prod_fit.xtc", "-n", "iface.ndx", "-d", "0.5",
        "-od", "mindist_dimer.xvg", "-on", "numcont_dimer.xvg", "-xvg", "none",
        stdin_text="sub1\nsub2\n", label="mindist between the subunits", quiet=True)
    md_dimer = xvg("mindist_dimer.xvg", "mindist")[:, 1] * 10.0
    nc_dimer = xvg("numcont_dimer.xvg", "contact count")[:, 1]

    per_pos = {}
    for p in CONV_IFACE:
        for k in (1, 2):
            other = "sub2" if k == 1 else "sub1"
            od = "mindist_pos%d_sub%d.xvg" % (p, k)
            gmx("mindist", "-s", tpr, "-f", "prod_fit.xtc", "-n", "iface.ndx",
                "-od", od, "-xvg", "none",
                stdin_text="pos%d_sub%d\n%s\n" % (p, k, other),
                label="mindist for position %d of subunit %d" % (p, k), quiet=True)
            v = xvg(od, "mindist position %d" % p)[:, 1] * 10.0
            per_pos["%d_sub%d" % (p, k)] = v

    hb = {}
    if pairs:
        gmx("distance", "-s", tpr, "-f", "prod_fit.xtc", "-n", "iface.ndx",
            "-select", ";".join('"%s"' % nm for nm, _, _, _ in pairs),
            "-oall", "hbond_dist.xvg", "-xvg", "none", label="polar pairs", quiet=True)
        D = xvg("hbond_dist.xvg", "gmx distance")
        if D.shape[1] - 1 != len(pairs):
            raise SystemExit("STOP: gmx distance returned %d columns for %d pairs"
                             % (D.shape[1] - 1, len(pairs)))
        for j, (nm, pretty, _, _) in enumerate(pairs):
            v = D[burn:, j + 1] * 10.0
            hb[pretty] = dict(mean=float(v.mean()), sd=float(v.std()),
                              occupancy_3_5=float((v <= 3.5).mean()))

    gmx("rms", "-s", tpr, "-f", "prod_fit.xtc", "-o", "rmsd.xvg", "-xvg", "none",
        *ndx_args(), stdin_text="SOLU\nSOLU\n", label="rmsd", quiet=True)
    rmsd = xvg("rmsd.xvg", "gmx rms")[burn:, 1] * 10.0

    def stats(v):
        v = np.asarray(v, dtype=float)
        return dict(mean=float(v.mean()), sd=float(v.std()),
                    p10=float(np.percentile(v, 10)), p90=float(np.percentile(v, 90)),
                    first=float(v[0]), last=float(v[-1]))

    t_ps = xvg("mindist_dimer.xvg", "mindist")[:, 0]
    sampled_ns = float((t_ps[-1] - t_ps[0]) / 1000.0)
    if sampled_ns <= 0:
        raise SystemExit("STOP: the trajectory spans %g ns, so its time axis is wrong"
                         % sampled_ns)
    out = dict(build=build, run=run, atoms=int(natoms), segments=prot,
               frames=int(n), burn=int(burn), ns=sampled_ns,
               ns_requested=float(NS_PER_BUILD),
               buried_per_subunit_A2=stats(buried[burn:]),
               buried_per_subunit_A2_full_series_mean=float(buried.mean()),
               mindist_subunits_A=stats(md_dimer[burn:]),
               contacts_0_5nm=stats(nc_dimer[burn:]),
               per_position_mindist_A={k: stats(v[burn:]) for k, v in per_pos.items()},
               polar_pairs=hb, rmsd_A=stats(rmsd),
               converted_interface_positions=CONV_IFACE,
               written=datetime.datetime.now().isoformat(timespec="seconds"))
    json.dump(out, open(os.path.join(RESULTS, "%s_report.json" % build), "w"), indent=2)
    print("  %d frames over %.1f ns, %d burned in" % (n, sampled_ns, burn))
    print("  buried interfacial area %.0f +/- %.0f A2 per subunit"
          % (out["buried_per_subunit_A2"]["mean"],
             out["buried_per_subunit_A2"]["sd"]))
    print("  inter-subunit contacts within 5 A %.0f +/- %.0f"
          % (out["contacts_0_5nm"]["mean"], out["contacts_0_5nm"]["sd"]))
    print("  backbone RMSD %.2f +/- %.2f A" % (out["rmsd_A"]["mean"], out["rmsd_A"]["sd"]))
    for k in sorted(per_pos):
        s = out["per_position_mindist_A"][k]
        print("  position %-9s min distance to the partner %.2f +/- %.2f A"
              % (k, s["mean"], s["sd"]))
    for pretty, s in sorted(hb.items()):
        print("  %-34s %.2f +/- %.2f A, within 3.5 A in %.0f per cent of frames"
              % (pretty, s["mean"], s["sd"], 100.0 * s["occupancy_3_5"]))
    return out


def equilibrate(run):
    if not os.path.isfile("step4.0_minimization.tpr"):
        gmx("grompp", "-f", "step4.0_minimization.mdp", "-o",
            "step4.0_minimization.tpr", "-c", "step3_input.gro", "-r",
            "step3_input.gro", "-p", "topol.top", "-n", "index.ndx",
            "-maxwarn", MAXWARN, label="grompp minimisation")
        must_exist("step4.0_minimization.tpr", "grompp minimisation")
    if not os.path.isfile("step4.0_minimization.gro"):
        gmx("mdrun", "-v", "-deffnm", "step4.0_minimization", label="minimisation")
        must_exist("step4.0_minimization.gro", "minimisation")
    prev = "step4.0_minimization"
    stages = sorted(glob.glob("step4.?_equilibration.mdp"))
    if not stages:
        raise SystemExit("STOP: no step4.*_equilibration.mdp in %s. A CHARMM-GUI "
                         "solution build writes at least one, so the download is "
                         "incomplete." % run)
    for mdp in stages:
        name = mdp[:-4]
        if os.path.isfile(name + ".gro"):
            prev = name
            continue
        gmx("grompp", "-f", mdp, "-o", name + ".tpr", "-c", prev + ".gro", "-r",
            "step3_input.gro", "-p", "topol.top", "-n", "index.ndx",
            "-maxwarn", MAXWARN, label="grompp " + name)
        must_exist(name + ".tpr", "grompp " + name)
        gmx("mdrun", "-v", "-deffnm", name, label=name)
        must_exist(name + ".gro", name)
        prev = name
    return prev


def produce(run, build, eq_final):
    mdp = "step5_production.mdp"
    if not os.path.isfile(mdp):
        raise SystemExit("STOP: no %s in %s; the CHARMM-GUI download is incomplete"
                         % (mdp, run))
    dt = None
    for L in open(mdp):
        m = re.match(r"\s*dt\s*=\s*([\d.]+)", L)
        if m:
            dt = float(m.group(1))
    if dt is None:
        raise SystemExit("STOP: could not read dt from %s, so the chunk length is "
                         "unknown" % mdp)
    nsteps = int(round(NS_PER_CHUNK * 1000.0 / dt))
    n_chunks = int(math.ceil(float(NS_PER_BUILD) / NS_PER_CHUNK))
    per_chunk = want_frames(mdp, NS_PER_CHUNK)
    drive_run = RUN_DIR[build]
    prev = eq_final
    for i in range(1, n_chunks + 1):
        name = "prod_%02d" % i
        if os.path.isfile(name + ".gro"):
            got = xtc_frames(name + ".xtc")
            if got is not None and got >= per_chunk - 1:
                prev = name
                continue
        if budget_left() < 0.6:
            print("  wall budget nearly spent, stopping cleanly after %d of %d chunks"
                  % (i - 1, n_chunks))
            return prev, i - 1, n_chunks
        t0 = time.time()
        resume = os.path.isfile(name + ".cpt") and not os.path.isfile(name + ".gro")
        if not os.path.isfile(name + ".tpr"):
            gmx("grompp", "-f", mdp, "-o", name + ".tpr", "-c", prev + ".gro", "-p",
                "topol.top", "-n", "index.ndx", "-maxwarn", MAXWARN,
                label="grompp " + name)
        must_exist(name + ".tpr", "grompp " + name)
        extra = ("-cpi", name + ".cpt") if resume else ()
        if resume:
            print("  %s was interrupted, resuming from its checkpoint" % name)
        gmx("mdrun", "-v", "-deffnm", name, "-nsteps", nsteps, *extra, label=name)
        must_exist(name + ".gro", name)
        got = xtc_frames(name + ".xtc")
        if got is None or got < per_chunk - 1:
            raise SystemExit("STOP: %s wrote %s frames and should hold %d. The chunk "
                             "is incomplete, and concatenating it would give a "
                             "trajectory shorter than it reports."
                             % (name, got, per_chunk))
        print("  %s %s done in %.1f min, %d frames"
              % (build, name, (time.time() - t0) / 60.0, got))
        stage_out(os.getcwd(), drive_run,
                  [name + ".gro", name + ".cpt", name + ".xtc"])
        drive_wait(why="chunk %d" % i)
        progress(**{build: dict(chunks_done=i, chunks_total=n_chunks)})
        prev = name
    return prev, n_chunks, n_chunks


def concatenate(run, n_chunks, per_chunk):
    """Join the chunks that finished, which may be fewer than the run asked for."""
    chunks = sorted(glob.glob("prod_[0-9][0-9].xtc"))
    if len(chunks) > n_chunks:
        raise SystemExit("STOP: %s holds %d chunk trajectories and the run asks for "
                         "%d: %s" % (os.getcwd(), len(chunks), n_chunks, chunks))
    if not chunks:
        raise SystemExit("STOP: %s holds no chunk trajectory" % os.getcwd())
    have_ns = len(chunks) * NS_PER_CHUNK
    counts = [xtc_frames(c) for c in chunks]
    short = [(c, n) for c, n in zip(chunks, counts)
             if n is None or n < per_chunk - 1]
    if short:
        raise SystemExit("STOP: %s hold fewer than the %d frames a chunk should "
                         "have. Concatenating them would produce a trajectory "
                         "shorter than it claims to be." % (short, per_chunk))
    for f in ("prod_all.xtc", "prod_nojump.xtc", "prod_fit.xtc"):
        if os.path.isfile(f):
            os.remove(f)
    offsets = "".join("%g\n" % (k * NS_PER_CHUNK * 1000.0) for k in range(len(chunks)))
    gmx("trjcat", "-f", *chunks, "-o", "prod_all.xtc", "-settime",
        stdin_text=offsets, label="trjcat")
    must_exist("prod_all.xtc", "trjcat")
    span = xtc_last_time("prod_all.xtc")
    if span is None or span < 0.9 * have_ns * 1000.0:
        raise SystemExit("STOP: prod_all.xtc ends at %s ps and the %d chunks that "
                         "finished should reach about %g ps. The time axis is wrong, "
                         "so every per-frame measurement downstream would be "
                         "mislabelled." % (span, len(chunks), have_ns * 1000.0))
    got = xtc_frames("prod_all.xtc")
    if got is None or got < sum(counts) - len(chunks):
        raise SystemExit("STOP: prod_all.xtc holds %s frames and the %d chunks hold "
                         "%d between them. The concatenation is incomplete."
                         % (got, len(chunks), sum(counts)))
    print("  concatenated %d chunks into %d frames, %d ns"
          % (len(chunks), got, have_ns))
    tpr = sorted(glob.glob("prod_[0-9][0-9].tpr"))[0]
    gmx("trjconv", "-s", tpr, "-f", "prod_all.xtc", "-o", "prod_nojump.xtc",
        *ndx_args(), "-pbc", "nojump", stdin_text="SOLU\n",
        label="trjconv nojump")
    must_exist("prod_nojump.xtc", "trjconv nojump")
    gmx("trjconv", "-s", tpr, "-f", "prod_nojump.xtc", "-o", "prod_fit.xtc",
        *ndx_args(), "-fit", "rot+trans", stdin_text="SOLU\nSOLU\n",
        label="trjconv fit")
    must_exist("prod_fit.xtc", "trjconv fit")
    gmx("trjconv", "-s", tpr, "-f", "prod_all.xtc", "-o", "prod_ref.pdb",
        *ndx_args(), "-dump", "0", stdin_text="SOLU\n", label="reference frame")
    must_exist("prod_ref.pdb", "reference frame")
    return tpr


def mdp_get(path, key):
    for l in open(path, errors="replace"):
        l = l.split(";")[0]
        if "=" in l and l.split("=")[0].strip().lower() == key.lower():
            return l.split("=", 1)[1].strip()
    return None


def mdp_set(path, key, value):
    """Rewrite one mdp setting in place, leaving every other line untouched."""
    out, hit = [], False
    for l in open(path, errors="replace"):
        body = l.split(";")[0]
        if "=" in body and body.split("=")[0].strip().lower() == key.lower():
            out.append("%-23s = %s\n" % (key, value)); hit = True
        else:
            out.append(l)
    if not hit:
        out.append("%-23s = %s\n" % (key, value))
    open(path, "w").writelines(out)

#@markdown ---
print("=" * 72)
print("PRE-FLIGHT: the two builds must differ only in the %d substitutions"
      % len(POSITIONS))
print("=" * 72)
ION = {"POT", "CLA", "SOD", "NA", "CL", "K", "KCL"}
pre = {}
for build in BUILD_ORDER:
    run = RUN_DIR[build]
    pdb = os.path.join(run, "step3_input.pdb")
    if not os.path.isfile(pdb):
        raise SystemExit("STOP: no %s. It is the gromacs/ folder from inside the "
                         "CHARMM-GUI download." % pdb)
    atoms = read_pdb_atoms(pdb)
    res = {}
    for ser, seg, ri, name, an in atoms:
        res.setdefault((seg, ri), name)
    ions = collections.Counter(n for n in res.values() if n in ION)
    ss = psf_disulfides(run)
    mdp = os.path.join(run, "step5_production.mdp")
    pre[build] = dict(atoms=len(atoms), ions=dict(ions), ss=len(ss),
                      prot=len(protein_segments(atoms)),
                      ref_t=mdp_get(mdp, "ref-t") or mdp_get(mdp, "ref_t"),
                      dt=mdp_get(mdp, "dt"),
                      waters=sum(1 for n in res.values() if n in ("TIP3", "HOH", "SOL")))
    print("  %-10s %7d atoms, %d protein segments, %d waters, ions %s, %d disulfides, "
          "ref_t %s" % (build, pre[build]["atoms"], pre[build]["prot"],
                        pre[build]["waters"], dict(ions) or "none", pre[build]["ss"],
                        pre[build]["ref_t"]))

for build in BUILD_ORDER:
    if pre[build]["ss"]:
        raise SystemExit("STOP: %s carries %d disulfide(s) in its topology. Part 1 "
                         "refused this structure if any SG pair was within 2.5 "
                         "angstrom, so CHARMM-GUI built a bond the protein does not "
                         "have, and it would hold the fold together by itself."
                         % (build, pre[build]["ss"]))
if len({pre[b]["prot"] for b in BUILD_ORDER}) != 1:
    raise SystemExit("STOP: the builds disagree on the number of protein segments: %s"
                     % {b: pre[b]["prot"] for b in BUILD_ORDER})
if len({tuple(sorted(pre[b]["ions"].items())) for b in BUILD_ORDER}) != 1:
    raise SystemExit("STOP: the builds carry different ion counts, %s. Every "
                     "substitution here is neutral to neutral, so the two systems "
                     "have the same net charge and should have been neutralised with "
                     "the same ions. A difference means the uploads were crossed or "
                     "the two CHARMM-GUI sessions did not use the same settings."
                     % {b: pre[b]["ions"] for b in BUILD_ORDER})
for key in ("ref_t", "dt"):
    vals = {pre[b][key] for b in BUILD_ORDER}
    if len(vals) != 1 or None in vals:
        raise SystemExit("STOP: the builds disagree on %s: %s. The comparison needs "
                         "the same conditions on both."
                         % (key, {b: pre[b][key] for b in BUILD_ORDER}))
TEMPERATURE = pre[BUILD_ORDER[0]]["ref_t"]
got_t = [float(x) for x in str(TEMPERATURE).replace(",", " ").split()]
if not got_t or max(abs(v - REF_T_WANT) for v in got_t) > 0.1:
    raise SystemExit("STOP: CHARMM-GUI built these systems with ref_t %s and this "
                     "series runs at %g K (%s). A run at another temperature is "
                     "not comparable to the rest of the line, and the two builds "
                     "matching each other does not make either of them right. Set "
                     "the temperature on CHARMM-GUI's dynamics page and download "
                     "again, or change ref_t_K in md/inputs/build_spec.json if the "
                     "series temperature really has moved."
                     % (TEMPERATURE, REF_T_WANT, REF_T_SOURCE))
spread = max(pre[b]["atoms"] for b in BUILD_ORDER) - min(pre[b]["atoms"] for b in BUILD_ORDER)
if spread > 3000:
    raise SystemExit("STOP: the builds differ by %d atoms, which is more than the "
                     "substitutions and their rebuilt side chains can account for, so "
                     "the water boxes differ. Rebuild both with the same edge "
                     "distance." % spread)
print("  ions agree across builds: %s"
      % (dict(pre[BUILD_ORDER[0]]["ions"]) or "none"))
print("  thermostat agrees across builds and with the spec: ref_t %s K"
      % TEMPERATURE)
print("  atom spread %d, within tolerance" % spread)

for build in BUILD_ORDER:
    mdp = os.path.join(RUN_DIR[build], "step5_production.mdp")
    dt = float(mdp_get(mdp, "dt"))
    want = int(round(TOTAL_NS * 1000.0 / dt / TARGET_FRAMES))
    want = max(100, want - want % 100)
    have = mdp_get(mdp, "nstxout-compressed")
    if have is None or int(have) != want:
        mdp_set(mdp, "nstxout-compressed", want)
        print("  %-10s nstxout-compressed %s -> %d  (%d frames over the planned %d "
              "ns, %d over the %d ns asked for now)"
              % (build, have, want, TOTAL_NS * 1000.0 / dt / want, TOTAL_NS,
                 NS_PER_BUILD * 1000.0 / dt / want, NS_PER_BUILD))
    else:
        print("  %-10s nstxout-compressed already %d" % (build, want))
progress(preflight="passed", ref_t=TEMPERATURE, atom_spread=spread,
         ions=pre[BUILD_ORDER[0]]["ions"])
print("\npre-flight passed, starting the run\n")

incomplete = []
for build in [BUILD]:
    run = RUN_DIR[build]
    rep = os.path.join(RESULTS, "%s_report.json" % build)
    print("\n" + "=" * 72)
    print("BUILD %s   %s" % (build.upper(), run))
    print("=" * 72)
    if os.path.isfile(rep) and json.load(open(rep)).get("ns", 0) >= 0.95 * NS_PER_BUILD:
        print("  already finished at %.1f ns" % json.load(open(rep))["ns"])
        continue
    required = ["step3_input.gro", "step3_input.pdb", "step3_input.psf", "topol.top",
                "step5_production.mdp", "index.ndx"]
    miss = [f for f in required if not os.path.isfile(os.path.join(run, f))]
    if miss:
        raise SystemExit("STOP: %s is missing %s. It should be the gromacs/ folder "
                         "from inside the CHARMM-GUI download." % (run, miss))
    if not globals().get("HAVE_GPU", False):
        raise SystemExit("STOP: build %s needs a GPU. Equilibration and production "
                         "both ask mdrun to offload, and section 2 found no usable "
                         "GPU, so this would fail hours from now with a message about "
                         "offload instead of about the runtime. Runtime > Change "
                         "runtime type > GPU, then run from the top." % build)
    natoms, prot = check_identity(run, build)
    n_chunks = int(math.ceil(float(NS_PER_BUILD) / NS_PER_CHUNK))
    per_chunk = want_frames(os.path.join(run, "step5_production.mdp"), NS_PER_CHUNK)
    work = os.path.join("/content/work", build)
    print("  a chunk should hold %d frames" % per_chunk)
    stage_in(run, work, n_chunks, per_chunk)
    os.chdir(work)
    eq = equilibrate(work)
    progress(**{build: dict(stage="equilibrated")})
    prev, done, total = produce(work, build, eq)
    if done == 0:
        incomplete.append((build, 0, total))
        continue
    tpr = concatenate(work, done, per_chunk)
    groups, pairs, segs = interface_groups(build)
    measure(run, build, tpr, groups, pairs, segs, natoms)
    stage_out(work, run, ["prod_fit.xtc", "prod_ref.pdb", os.path.basename(tpr),
                          "iface.ndx", "sasa_dimer.xvg", "sasa_sub1.xvg",
                          "sasa_sub2.xvg", "mindist_dimer.xvg", "numcont_dimer.xvg",
                          "rmsd.xvg"]
              + (["hbond_dist.xvg"] if pairs else [])
              + ["mindist_pos%d_sub%d.xvg" % (p, k) for p in CONV_IFACE for k in (1, 2)])
    drive_wait(why="the analysis files")
    print("  copied the fitted trajectory and its analysis files to Drive")
    if done < total:
        incomplete.append((build, done, total))
    progress(**{build: dict(stage="measured", chunks_done=done, chunks_total=total)})

os.chdir("/content")
drive_verify()
if incomplete:
    for b, d, t in incomplete:
        print("%s reached %d of %d chunks" % (b, d, t))
    print("\nThe wall budget ran out before %s reached %d ns. Nothing is lost: the "
          "chunks that finished are measured and reported, and opening this notebook "
          "again and pressing Run all continues from the last one. The cached GROMACS "
          "means the next session starts in about a minute." % (BUILD, NS_PER_BUILD))
    print("Part 3 scores what is there: it can stop the series on a short run, and it "
          "withholds a pass until the full %d ns." % NS_PER_BUILD)
print("\n%s finished. Run the other build in its own notebook, then open "
      "1433_rQTY_MD_3_compare.ipynb, which applies the kill criterion."
      % BUILD.upper())


## Done

The report for this build is in `md/results/<build>_report.json` on Drive, with the
fitted trajectory, its reference frame, the index file and every xvg beside the
CHARMM-GUI folder it came from. Nothing of value is left on the virtual machine.

The report carries the length actually sampled, so a session that ran out of wall
budget still produces one. Part 3 scores what is there: it can stop the series on a
short run and it withholds a pass until the full length.

Run the other build in its own notebook, then open `1433_rQTY_MD_3_compare.ipynb`.
